In [1]:
# Customize your practice plan here
my_custom_plan = generate_practice_plan(
    total_time=75,  # Change this to your desired time
    categories=['warmup', 'technical', 'small_sided_games', 'cool_down']  # Customize or set to None for all
)

NameError: name 'generate_practice_plan' is not defined

## Setup and Load Data

In [1]:
# Import required libraries
import pandas as pd
import random
from pathlib import Path

# Define paths
DATA_PATH = Path.home() / "Downloads"

# Define CSV file paths
csv_files = {
    'warmup': DATA_PATH / "warmup_drills.csv",
    'cool_down': DATA_PATH / "cool_down_drills.csv",
    'conditioning_with_ball': DATA_PATH / "conditioning_with_ball.csv",
    'small_sided_games': DATA_PATH / "small_sided_games.csv",
    'tactical': DATA_PATH / "tactical_drills.csv",
    'technical': DATA_PATH / "technical_drills.csv"
}

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Load and combine all drill data
all_drills = []

for category, file_path in csv_files.items():
    df = pd.read_csv(file_path)
    df['category'] = category
    all_drills.append(df)

# Concatenate all dataframes
drills_raw = pd.concat(all_drills, ignore_index=True)

# Assign duration to each drill based on category
duration_map = {
    'warmup': 15,
    'cool_down': 15,
    'conditioning_with_ball': 15,
    'small_sided_games': 25,
    'tactical': 20,
    'technical': 25
}

drills_raw['duration'] = drills_raw['category'].map(duration_map)

print(f"✓ Loaded {len(drills_raw)} drills from {len(csv_files)} categories")
print(f"\nDrills by category:")
print(drills_raw['category'].value_counts())

✓ Loaded 30 drills from 6 categories

Drills by category:
category
warmup                    5
cool_down                 5
conditioning_with_ball    5
small_sided_games         5
tactical                  5
technical                 5
Name: count, dtype: int64


## Practice Plan Generator Function

In [3]:
def generate_practice_plan(total_time, categories=None, drills_df=None):
    """
    Generate a practice plan based on time constraints and desired categories.
    
    Parameters:
    -----------
    total_time : int
        Total practice time in minutes
    categories : list, optional
        List of drill categories to include. If None, uses all categories.
    drills_df : DataFrame, optional
        DataFrame containing drills. If None, uses drills_raw.
    
    Returns:
    --------
    DataFrame with selected drills and practice plan summary
    """
    if drills_df is None:
        drills_df = drills_raw
    
    # Use all categories if none specified
    if categories is None:
        categories = drills_df['category'].unique().tolist()
    
    # Filter to requested categories
    available_drills = drills_df[drills_df['category'].isin(categories)].copy()
    
    # Track selected drills and time used
    selected_drills = []
    time_used = 0
    categories_used = set()
    
    # Shuffle categories for variety
    category_order = random.sample(categories, len(categories))
    
    # Try to include at least one drill from each requested category
    for cat in category_order:
        cat_drills = available_drills[available_drills['category'] == cat]
        if len(cat_drills) > 0:
            drill = cat_drills.sample(1).iloc[0]
            drill_time = drill['duration']
            
            if time_used + drill_time <= total_time:
                selected_drills.append(drill)
                time_used += drill_time
                categories_used.add(cat)
                # Remove selected drill from available
                available_drills = available_drills[available_drills.index != drill.name]
    
    # Fill remaining time with random drills
    while time_used < total_time and len(available_drills) > 0:
        drill = available_drills.sample(1).iloc[0]
        drill_time = drill['duration']
        
        if time_used + drill_time <= total_time:
            selected_drills.append(drill)
            time_used += drill_time
            categories_used.add(drill['category'])
            available_drills = available_drills[available_drills.index != drill.name]
        else:
            break
    
    # Create results DataFrame
    if selected_drills:
        plan_df = pd.DataFrame(selected_drills)
        
        # Print summary
        print("=" * 70)
        print("PRACTICE PLAN SUMMARY")
        print("=" * 70)
        print(f"Total Time: {time_used}/{total_time} minutes")
        print(f"Number of Drills: {len(plan_df)}")
        print(f"Categories Included: {', '.join(sorted(categories_used))}")
        print("\n" + "=" * 70)
        print("DRILL SEQUENCE")
        print("=" * 70)
        
        cumulative_time = 0
        for idx, drill in enumerate(selected_drills):
            cumulative_time += drill['duration']
            print(f"\n{idx + 1}. [{drill['category'].upper().replace('_', ' ')}] - {drill['duration']} min (Total: {cumulative_time} min)")
            
            # Print drill details if columns exist
            for col in ['name', 'title', 'drill_name', 'Name', 'Title']:
                if col in drill.index and pd.notna(drill[col]):
                    print(f"   Name: {drill[col]}")
                    break
            
            for col in ['description', 'desc', 'Description']:
                if col in drill.index and pd.notna(drill[col]):
                    desc = str(drill[col])[:150]
                    print(f"   Description: {desc}...")
                    break
        
        print("\n" + "=" * 70)
        
        return plan_df
    else:
        print("⚠️ Could not generate a plan with the given constraints.")
        return pd.DataFrame()

print("✓ Practice plan generator function loaded")

✓ Practice plan generator function loaded


## Available Drill Categories

The following drill categories are available with their default durations:

| Category | Duration (minutes) |
|----------|-------------------|
| **warmup** | 15 |
| **cool_down** | 15 |
| **conditioning_with_ball** | 15 |
| **small_sided_games** | 25 |
| **tactical** | 20 |
| **technical** | 25 |

## Example 1: Full 60-Minute Practice

In [4]:
# Generate a 60-minute practice with all categories
plan1 = generate_practice_plan(total_time=60)

PRACTICE PLAN SUMMARY
Total Time: 60/60 minutes
Number of Drills: 3
Categories Included: cool_down, tactical, technical

DRILL SEQUENCE

1. [TACTICAL] - 20 min (Total: 20 min)
   Name: Wide Channel Decision Play
   Description: Develop patterns down the flanks and final-third choices....

2. [TECHNICAL] - 25 min (Total: 45 min)
   Name: First Touch Direction
   Description: Receive across body and play into target quickly....

3. [COOL DOWN] - 15 min (Total: 60 min)
   Name: Breathwork + Touches
   Description: Slow juggling or touches paired with controlled breathing....



## Example 2: 90-Minute Technical & Tactical Focus

In [5]:
# Generate a 90-minute practice focusing on technical and tactical drills
plan2 = generate_practice_plan(
    total_time=90, 
    categories=['warmup', 'technical', 'tactical', 'small_sided_games', 'cool_down']
)

PRACTICE PLAN SUMMARY
Total Time: 85/90 minutes
Number of Drills: 4
Categories Included: small_sided_games, tactical, technical, warmup

DRILL SEQUENCE

1. [TACTICAL] - 20 min (Total: 20 min)
   Name: Wide Channel Decision Play
   Description: Develop patterns down the flanks and final-third choices....

2. [WARMUP] - 15 min (Total: 35 min)
   Name: Two-Cone Rondo Warmup
   Description: Simple rondo to get touches and warm up movement patterns....

3. [SMALL SIDED GAMES] - 25 min (Total: 60 min)
   Name: 4v4 + 2 Neutrals
   Description: Neutrals help maintain possession and find central pockets....

4. [TECHNICAL] - 25 min (Total: 85 min)
   Name: Defensive Jockeying
   Description: Players learn to angle attacker away and time the tackle....



## Example 3: 45-Minute Conditioning Session

In [6]:
# Generate a 45-minute conditioning-focused practice
plan3 = generate_practice_plan(
    total_time=45,
    categories=['warmup', 'conditioning_with_ball', 'small_sided_games', 'cool_down']
)

PRACTICE PLAN SUMMARY
Total Time: 45/45 minutes
Number of Drills: 3
Categories Included: conditioning_with_ball, cool_down, warmup

DRILL SEQUENCE

1. [CONDITIONING WITH BALL] - 15 min (Total: 15 min)
   Name: High-Tempo Passing Grid
   Description: Short sharp passing in tight space with constant movement....

2. [COOL DOWN] - 15 min (Total: 30 min)
   Name: Walking Technical Reset
   Description: Players walk laps touching the ball every step to calm tempo....

3. [WARMUP] - 15 min (Total: 45 min)
   Name: Dynamic Ball Mastery
   Description: Light dynamic movements with ball touches between each cone....



## Create Your Own Custom Practice Plan

Use the cell below to generate your own custom practice plan by adjusting:
- **`total_time`**: Total minutes for the practice session
- **`categories`**: List of drill categories to include (or leave as `None` for all categories)

In [ ]:
# Customize your practice plan here
my_custom_plan = generate_practice_plan(
    total_time=75,  # Change this to your desired time
    categories=['warmup', 'technical', 'small_sided_games', 'cool_down']  # Customize or set to None for all
)

## Create Your Own Custom Practice Plan

Use the cell below to generate your own custom practice plan by adjusting:
- **`total_time`**: Total minutes for the practice session
- **`categories`**: List of drill categories to include (or leave as `None` for all categories)

In [ ]:
# Generate a 45-minute conditioning-focused practice
plan3 = generate_practice_plan(
    total_time=45,
    categories=['warmup', 'conditioning_with_ball', 'small_sided_games', 'cool_down']
)

## Example 3: 45-Minute Conditioning Session

In [ ]:
# Generate a 90-minute practice focusing on technical and tactical drills
plan2 = generate_practice_plan(
    total_time=90, 
    categories=['warmup', 'technical', 'tactical', 'small_sided_games', 'cool_down']
)

## Example 2: 90-Minute Technical & Tactical Focus

In [ ]:
# Generate a 60-minute practice with all categories
plan1 = generate_practice_plan(total_time=60)

## Example 1: Full 60-Minute Practice

## Available Drill Categories

The following drill categories are available with their default durations:

| Category | Duration (minutes) |
|----------|-------------------|
| **warmup** | 15 |
| **cool_down** | 15 |
| **conditioning_with_ball** | 15 |
| **small_sided_games** | 25 |
| **tactical** | 20 |
| **technical** | 25 |

In [ ]:
def generate_practice_plan(total_time, categories=None, drills_df=None):
    """
    Generate a practice plan based on time constraints and desired categories.
    
    Parameters:
    -----------
    total_time : int
        Total practice time in minutes
    categories : list, optional
        List of drill categories to include. If None, uses all categories.
    drills_df : DataFrame, optional
        DataFrame containing drills. If None, uses drills_raw.
    
    Returns:
    --------
    DataFrame with selected drills and practice plan summary
    """
    if drills_df is None:
        drills_df = drills_raw
    
    # Use all categories if none specified
    if categories is None:
        categories = drills_df['category'].unique().tolist()
    
    # Filter to requested categories
    available_drills = drills_df[drills_df['category'].isin(categories)].copy()
    
    # Track selected drills and time used
    selected_drills = []
    time_used = 0
    categories_used = set()
    
    # Shuffle categories for variety
    category_order = random.sample(categories, len(categories))
    
    # Try to include at least one drill from each requested category
    for cat in category_order:
        cat_drills = available_drills[available_drills['category'] == cat]
        if len(cat_drills) > 0:
            drill = cat_drills.sample(1).iloc[0]
            drill_time = drill['duration']
            
            if time_used + drill_time <= total_time:
                selected_drills.append(drill)
                time_used += drill_time
                categories_used.add(cat)
                # Remove selected drill from available
                available_drills = available_drills[available_drills.index != drill.name]
    
    # Fill remaining time with random drills
    while time_used < total_time and len(available_drills) > 0:
        drill = available_drills.sample(1).iloc[0]
        drill_time = drill['duration']
        
        if time_used + drill_time <= total_time:
            selected_drills.append(drill)
            time_used += drill_time
            categories_used.add(drill['category'])
            available_drills = available_drills[available_drills.index != drill.name]
        else:
            break
    
    # Create results DataFrame
    if selected_drills:
        plan_df = pd.DataFrame(selected_drills)
        
        # Print summary
        print("=" * 70)
        print("PRACTICE PLAN SUMMARY")
        print("=" * 70)
        print(f"Total Time: {time_used}/{total_time} minutes")
        print(f"Number of Drills: {len(plan_df)}")
        print(f"Categories Included: {', '.join(sorted(categories_used))}")
        print("\n" + "=" * 70)
        print("DRILL SEQUENCE")
        print("=" * 70)
        
        cumulative_time = 0
        for idx, drill in enumerate(selected_drills):
            cumulative_time += drill['duration']
            print(f"\n{idx + 1}. [{drill['category'].upper().replace('_', ' ')}] - {drill['duration']} min (Total: {cumulative_time} min)")
            
            # Print drill details if columns exist
            for col in ['name', 'title', 'drill_name', 'Name', 'Title']:
                if col in drill.index and pd.notna(drill[col]):
                    print(f"   Name: {drill[col]}")
                    break
            
            for col in ['description', 'desc', 'Description']:
                if col in drill.index and pd.notna(drill[col]):
                    desc = str(drill[col])[:150]
                    print(f"   Description: {desc}...")
                    break
        
        print("\n" + "=" * 70)
        
        return plan_df
    else:
        print("⚠️ Could not generate a plan with the given constraints.")
        return pd.DataFrame()

print("✓ Practice plan generator function loaded")

## Practice Plan Generator Function

In [ ]:
# Load and combine all drill data
all_drills = []

for category, file_path in csv_files.items():
    df = pd.read_csv(file_path)
    df['category'] = category
    all_drills.append(df)

# Concatenate all dataframes
drills_raw = pd.concat(all_drills, ignore_index=True)

# Assign duration to each drill based on category
duration_map = {
    'warmup': 15,
    'cool_down': 15,
    'conditioning_with_ball': 15,
    'small_sided_games': 25,
    'tactical': 20,
    'technical': 25
}

drills_raw['duration'] = drills_raw['category'].map(duration_map)

print(f"✓ Loaded {len(drills_raw)} drills from {len(csv_files)} categories")
print(f"\nDrills by category:")
print(drills_raw['category'].value_counts())

In [3]:
# Import required libraries
import pandas as pd
import random
from pathlib import Path

# Define paths
DATA_PATH = Path.home() / "Downloads"

# Define CSV file paths
csv_files = {
    'warmup': DATA_PATH / "warmup_drills.csv",
    'cool_down': DATA_PATH / "cool_down_drills.csv",
    'conditioning_with_ball': DATA_PATH / "conditioning_with_ball.csv",
    'small_sided_games': DATA_PATH / "small_sided_games.csv",
    'tactical': DATA_PATH / "tactical_drills.csv",
    'technical': DATA_PATH / "technical_drills.csv"
}

print("✓ Setup complete")

✓ Setup complete


## Setup and Load Data

# Interactive Practice Plan Generator

This notebook generates customized soccer practice plans based on time constraints and desired drill categories.